# LangChain + LangGraph: A Small Multi-Agent Assistant

**IAN 620 · Practical tutorial**

A user asks a question. A local language model helps choose a route, and LangGraph sends the request to the appropriate agent.

- **Weather agent:** an API placeholder, currently returning sample weather.
- **Jobs agent:** an API placeholder, currently returning fictional job listings.
- **Profit agent:** calculates real arithmetic from the numbers provided.
- **General agent:** uses the local LLM for other questions.

This notebook also supports requests involving **more than one agent**, such as "Weather and jobs in Greensboro." The API-backed results are **demonstrations, not live facts**.

No paid LLM API key is needed. The first model download may take time. Run cells from top to bottom in Google Colab.

## 1. Install the libraries

**LangChain** connects the language model and tools. **LangGraph** controls which agent runs next. **Transformers** loads a small instruction-tuned LLM locally.

Colab's free CPU runtime can work, although a GPU runtime is faster.

In [ ]:
%pip -q install -U langchain langgraph langchain-huggingface transformers accelerate

## 2. Load a local LLM

We use a compact Qwen instruction model. The code sets deterministic generation for more repeatable demonstrations. You can replace the model ID with another compatible text-generation model.

In [ ]:
from langchain_huggingface import HuggingFacePipeline

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 120,
        "do_sample": False,
        "return_full_text": False,
    },
)
print("LLM loaded:", MODEL_ID)

## 3. LangChain tools: weather, jobs, and profit

A **tool** is a Python function that an agent can use. The `@tool` decorator tells LangChain the function's name, parameters, and description.

For now the two external API calls are **commented out**. The response from each is a clearly labeled example, so the notebook runs without an API key. To connect a real API later, replace the sample data with validated API responses.

The profit calculation does not require an API or LLM: **profit = (selling price − cost price) × quantity**.

In [ ]:
from langchain_core.tools import tool

@tool
def weather_tool(location: str) -> str:
    """Get weather information for a city (sample response; API disabled)."""
    # Example integration (requires a real endpoint and error handling):
    # import requests
    # response = requests.get(WEATHER_API_URL, params={"city": location, "key": API_KEY}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"SAMPLE WEATHER ONLY — not a live forecast for {location}: "
        "72°F, partly cloudy, light breeze."
    )

@tool
def jobs_tool(location: str) -> str:
    """Find recent job postings in a city (fictional sample; API disabled)."""
    # Example integration (requires a job-search provider and authentication):
    # import requests
    # response = requests.get(JOBS_API_URL, params={"location": location}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"FICTIONAL SAMPLE JOBS for {location} — not current openings:\\n"
        "1. Data Analyst — Example Analytics Co.\\n"
        "2. Python Developer — Demo Software Inc.\\n"
        "3. AI Engineer — Sample Technology Group"
    ).replace("\\n", "\n")

@tool
def profit_tool(selling_price: float, cost_price: float, quantity: int = 1) -> str:
    """Calculate total profit from selling price, cost price, and quantity."""
    if quantity < 0:
        return "Quantity must be zero or positive."
    total_revenue = selling_price * quantity
    total_cost = cost_price * quantity
    profit = total_revenue - total_cost
    margin = (profit / total_revenue * 100) if total_revenue else 0
    return (
        f"Revenue: ${total_revenue:,.2f}; Cost: ${total_cost:,.2f}; "
        f"Profit: ${profit:,.2f}; Profit margin: {margin:.1f}%."
    )

print(weather_tool.invoke({"location": "Greensboro"}))
print(jobs_tool.invoke({"location": "Greensboro"}))
print(profit_tool.invoke({"selling_price": 75, "cost_price": 50, "quantity": 10}))

## 4. First, understand the state

LangGraph passes a **state dictionary** between nodes. In this notebook it stores the user's question, the selected agents, each agent's output, and the final answer.

We keep the graph straightforward: **router → selected agent → router/next agent → final response**. Each agent can be called only once for a single question, so the workflow has a clear stopping point.

In [ ]:
from typing import TypedDict

class AssistantState(TypedDict):
    question: str
    routes: list[str]
    results: list[str]
    answer: str

## 5. Route the question with the LLM

The language model receives an instruction to choose one of four routes. We also recognize obvious keywords, which makes demonstrations with the small local model more dependable.

For a combined request, the keyword check collects multiple routes. For less obvious requests, the LLM provides a suggested route. In a production system, you would use structured output and stronger validation.

In [ ]:
import re

ROUTE_LABELS = ("weather", "jobs", "profit", "general")

def choose_routes(question: str) -> list[str]:
    text = question.lower()
    routes = []

    if any(word in text for word in ("weather", "temperature", "forecast", "rain", "sunny")):
        routes.append("weather")
    if any(word in text for word in ("job", "hiring", "vacanc", "opening", "career", "position")):
        routes.append("jobs")
    if any(word in text for word in ("profit", "revenue", "margin", "selling price", "cost price")):
        routes.append("profit")

    if routes:
        return routes

    prompt = (
        "Classify the user question. Reply with ONE lowercase word only: "
        "weather, jobs, profit, or general.\n"
        f"Question: {question}\nCategory:"
    )
    response = llm.invoke(prompt).strip().lower()
    match = re.search(r"\b(weather|jobs|profit|general)\b", response)
    return [match.group(1)] if match else ["general"]

def route_node(state: AssistantState) -> dict:
    routes = choose_routes(state["question"])
    print("Selected agents:", routes)
    return {"routes": routes}

## 6. Implement the three specialist agents

Each node uses a LangChain tool. The demonstration location defaults to Greensboro if no city is identified. The profit agent extracts numbers from a clearly formatted example; it asks for more details if the user does not provide enough information.

**Example profit prompt:** "Calculate profit: selling price 75, cost price 50, quantity 10."

In [ ]:
def find_location(question: str) -> str:
    match = re.search(r"\b(?:in|for|near)\s+([A-Za-z ]+?)(?=[,?.!]|\s+and\s+|$)", question, re.I)
    return match.group(1).strip() if match else "Greensboro"

def weather_agent(state: AssistantState) -> dict:
    result = weather_tool.invoke({"location": find_location(state["question"])})
    return {"results": state["results"] + [result]}

def jobs_agent(state: AssistantState) -> dict:
    result = jobs_tool.invoke({"location": find_location(state["question"])})
    return {"results": state["results"] + [result]}

def extract_number(question: str, names: str):
    pattern = rf"(?:{names})\s*(?:is|=|:|of)?\s*\$?(-?\d+(?:\.\d+)?)"
    match = re.search(pattern, question, re.I)
    return float(match.group(1)) if match else None

def profit_agent(state: AssistantState) -> dict:
    question = state["question"]
    selling = extract_number(question, r"selling\s*price|sale\s*price")
    cost = extract_number(question, r"cost\s*price|unit\s*cost|cost")
    count = extract_number(question, r"quantity|units|items")
    if selling is None or cost is None:
        result = (
            "To calculate profit, specify both prices. Example: "
            "'selling price 75, cost price 50, quantity 10'."
        )
    elif count is not None and (count < 0 or not count.is_integer()):
        result = "Quantity must be a nonnegative whole number."
    else:
        result = profit_tool.invoke({
            "selling_price": selling,
            "cost_price": cost,
            "quantity": int(count) if count is not None else 1,
        })
    return {"results": state["results"] + [result]}

def general_agent(state: AssistantState) -> dict:
    prompt = (
        "Answer this question briefly and clearly. If you are unsure, say so.\n"
        f"Question: {state['question']}\nAnswer:"
    )
    return {"results": state["results"] + [llm.invoke(prompt).strip()]}

## 7. Build the LangGraph workflow

The router chooses a list of agents. The dispatcher sends the state to the next one. After an agent finishes, the graph checks whether another route remains. Finally, it combines the results.

This is a **sequential multi-agent graph**. It is easier to inspect than a parallel graph and avoids state merging complexities.

In [ ]:
from langgraph.graph import StateGraph, START, END

def next_agent(state: AssistantState) -> str:
    completed = len(state["results"])
    if completed < len(state["routes"]):
        return state["routes"][completed]
    return "finish"

def finish_node(state: AssistantState) -> dict:
    return {"answer": "\n\n".join(state["results"])}

builder = StateGraph(AssistantState)
builder.add_node("router", route_node)
builder.add_node("weather", weather_agent)
builder.add_node("jobs", jobs_agent)
builder.add_node("profit", profit_agent)
builder.add_node("general", general_agent)
builder.add_node("finish", finish_node)

builder.add_edge(START, "router")
destinations = {name: name for name in ROUTE_LABELS}
destinations["finish"] = "finish"
builder.add_conditional_edges("router", next_agent, destinations)

for name in ROUTE_LABELS:
    builder.add_conditional_edges(name, next_agent, destinations)

builder.add_edge("finish", END)
assistant = builder.compile()
print("Graph compiled.")

## 8. Try three questions

The first two answers are examples rather than current information. The third calculates from your supplied numbers.

In [ ]:
def ask(question: str) -> str:
    state = assistant.invoke({
        "question": question,
        "routes": [],
        "results": [],
        "answer": "",
    })
    print("\nQUESTION:", question)
    print("ANSWER:\n", state["answer"])
    print("-" * 65)
    return state["answer"]

weather_answer = ask("What is the weather in Greensboro?")
jobs_answer = ask("Find recent jobs in Greensboro.")
profit_answer = ask("Calculate profit: selling price 75, cost price 50, quantity 10.")

## 9. A question that needs two agents

The next prompt mentions both weather and jobs, so the graph visits both specialist nodes before finishing. The returned weather and job details are still **dummy data**.

In [ ]:
combined_answer = ask("What is the weather in Greensboro and are there any recent jobs in Greensboro?")

## 10. Enter your own question

Run this cell and change the text. Try a general question, a profit calculation, or a combined request.

In [ ]:
question = "Calculate profit: selling price 120, cost price 85, quantity 4."
answer = ask(question)

## 11. Optional: inspect the graph

LangGraph can draw the nodes and their possible connections in a notebook.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(assistant.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Diagram unavailable:", exc)
    print(assistant.get_graph().draw_mermaid())

## 12. How to turn this into a real agent system

1. Replace the **commented weather API** with a weather provider that returns location, timestamp, and measurements.
2. Replace the **commented jobs API** with an authorized jobs endpoint. Never label fictional examples as real job openings.
3. For more flexible profit questions, use the LLM to extract a validated structured object, then let Python do the arithmetic.
4. Add tool error handling, timeouts, API credentials via Colab secrets, a maximum agent count, and tests.
5. For more advanced workflows, add retrieval, human approval, or an evaluator node.

**Checkpoint:** Can you explain which part is the LLM, which parts are LangChain tools, and how LangGraph decides which node runs next?

Official references: [LangChain](https://python.langchain.com/docs/introduction/) · [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) · [Hugging Face integration](https://reference.langchain.com/python/langchain-huggingface/llms/huggingface_pipeline/HuggingFacePipeline)